# Atlas Notochord Cell Abundance Comparison

Compare raw embryo-level cell abundance trends across atlases for notochord-related cell types.

In [ ]:
hdf5_lib <- "/net/gs/vol3/software/modules-sw/hdf5/1.14.3/Linux/Ubuntu22.04/x86_64/lib/libhdf5.so.310"
if (file.exists(hdf5_lib)) dyn.load(hdf5_lib)

suppressPackageStartupMessages({
  library(monocle3)
  library(BPCells)
  library(hooke)
  library(dplyr)
  library(tidyr)
  library(tibble)
  library(ggplot2)
  library(stringr)
  library(purrr)
})

## Parameters

In [ ]:
temp_path <- "/net/trapnell/vol1/home/nlammers/tmp_files/nobackup/"
seahub_root <- "/net/seahub_zfish/vol1/data/reference_cds/"

dataset_specs <- tribble(
  ~dataset, ~path, ~harmonization_family,
  "v2.3.0", file.path(seahub_root, "v2.3.0"), "v2.3.0",
  "v3.1.0", file.path(seahub_root, "v3.1.0"), "v3.1.0",
  "v3.1.0 GENE8 run 1", "/net/seahub_zfish/vol1/data/seahub_rna_processing/portal_inputs/v3.1.0/mcclintock/GENE8/run_1/filter_embryos/embryo_filtered_cds", "v3.1.0",
  "v3.1.0 GENE9 run 1", "/net/seahub_zfish/vol1/data/seahub_rna_processing/portal_inputs/v3.1.0/mcclintock/GENE9/run_1/filter_embryos/embryo_filtered_cds", "v3.1.0",
  "v3.1.0 GENE10 run 1", "/net/seahub_zfish/vol1/data/seahub_rna_processing/portal_inputs/v3.1.0/mcclintock/GENE10/run_1/filter_embryos/embryo_filtered_cds", "v3.1.0",
  # GENE14 (was mislabelled "GENE11"; path is GENE14, no GENE11 exists under
  # mcclintock). It is the ciliopathy panel (cep290/b9d2/sspo/...) and was not part
  # of the sheath power comparison; its count cache has not been built, so it is
  # disabled here to keep the read-back cell runnable. Uncomment and run the active
  # atlas count loop for it if you want it back in.
  # "v3.1.0 GENE14 run 1", "/net/seahub_zfish/vol1/data/seahub_rna_processing/portal_inputs/v3.1.0/mcclintock/GENE14/run_1/filter_embryos/embryo_filtered_cds", "v3.1.0",
  # GENE7 covers the full 24/30/36 hpf collection panel at depth. It is a
  # temperature-shift experiment, but the ctrl_labels filter keeps only the plain
  # "ctrl-inj" embryos, which are all 28C -- the shifted arms are labelled
  # "ctrl-inj,hot"/"ctrl-inj,cold" and do not match, so no extra temp filter is needed.
  "v3.1.0 GENE7 run 1", "/net/seahub_zfish/vol1/data/seahub_rna_processing/portal_inputs/v3.1.0/mcclintock/GENE7/run_1/filter_embryos/embryo_filtered_cds", "v3.1.0",
  # GENE11 is the most representative OLD-method dataset. It lives under v3.0.1 but
  # uses the v3.1.0-style cell-type vocabulary, so harmonization_family is "v3.1.0".
  # 28C throughout; ctrl_labels keeps only the "ctrl-inj" controls. Overlaps the
  # 24/30/36 panel at 36 hpf only (its stages are 36/48/72).
  "v3.0.1 GENE11 run 1", "/net/seahub_zfish/vol1/data/seahub_rna_processing/portal_inputs/v3.0.1/mcclintock/GENE11/run_1/filter_embryos/embryo_filtered_cds", "v3.1.0"
)

# Add additional atlas directories as new rows in dataset_specs.
datasets_to_compare <- dataset_specs$dataset

# out_dir <- "/Users/nick/Library/CloudStorage/GoogleDrive-nlammers@uw.edu/My Drive/projects/morphseq/results/20260615"
out_dir <- "/net/trapnell/vol1/home/nlammers/projects/data/morphseq/results/nlammers/20260615"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
count_cache_dir <- file.path(out_dir, "atlas_count_cache")
dir.create(count_cache_dir, recursive = TRUE, showWarnings = FALSE)

active_dataset <- datasets_to_compare[[5]]
load_metadata_only <- TRUE
force_recount_cell_counts <- FALSE
plot_hpf_min <- 24
plot_hpf_values <- c(24, 36, 48, 60, 72)
plot_hpf_label <- paste(plot_hpf_values, collapse = ", ")

hpf_range <- c(12, 72)
ctrl_labels <- c("EtOH", "DMSO", "ctrl-inj", "reference", "ctrl-uninj", "novehicle")

sample_group_candidates <- c("embryo_ID", "embryo_id", "embryo", "sample", "sample_id")
timepoint_candidates <- c("timepoint", "hpf", "stage_hpf", "age_hpf", "hours_post_fertilization")
cell_group_candidates <- c("cell_type", "cell_type_broad")

notochord_pattern <- regex("notochord|hypochord", ignore_case = TRUE)
theme_set(theme_bw(base_size = 18))

## Helpers

In [ ]:
first_present <- function(candidates, columns, label) {
  hit <- candidates[candidates %in% columns]
  if (length(hit) == 0) {
    stop(sprintf("Could not find %s. Tried: %s", label, paste(candidates, collapse = ", ")))
  }
  hit[[1]]
}

parse_hpf <- function(x) {
  if (is.numeric(x)) return(as.numeric(x))
  as.numeric(stringr::str_extract(as.character(x), "[0-9]+\\.?[0-9]*"))
}

dataset_base_path <- function(dataset_name) {
  match_idx <- match(dataset_name, dataset_specs$dataset)
  if (!is.na(match_idx)) return(dataset_specs$path[[match_idx]])
  dataset_name
}

resolve_cds_path <- function(dataset_name, root = seahub_root) {
  dataset_path <- dataset_base_path(dataset_name)
  base_dir <- if (dir.exists(dataset_path)) dataset_path else file.path(root, dataset_path)
  dataset_label <- basename(normalizePath(base_dir, mustWork = FALSE))
  candidates <- unique(c(
    file.path(base_dir, "reference_cds"),
    file.path(base_dir, "projected_cds"),
    file.path(base_dir, paste0(dataset_label, "_projected_cds")),
    file.path(base_dir, paste0(dataset_label, "_projected_cds_", dataset_label)),
    file.path(base_dir, paste0("reference_projected_cds_", dataset_label)),
    base_dir
  ))
  existing <- candidates[dir.exists(candidates)]
  existing_with_rds <- existing[file.exists(file.path(existing, "cds_object.rds"))]
  if (length(existing_with_rds) > 0) return(existing_with_rds[[1]])
  if (length(existing) > 0) return(existing[[1]])

  if (dir.exists(base_dir)) {
    nested <- list.dirs(base_dir, recursive = TRUE, full.names = TRUE)
    nested <- nested[str_detect(basename(nested), regex("cds|monocle|projected", ignore_case = TRUE))]
    if (length(nested) > 0) return(nested[[1]])
  }

  stop(sprintf("Could not resolve CDS path for %s under %s", dataset_name, root))
}

load_atlas_cds <- function(dataset_name, metadata_only = load_metadata_only) {
  message("Loading ", dataset_name)
  cds_path <- resolve_cds_path(dataset_name)
  message("  CDS path: ", cds_path)

  if (metadata_only) {
    cds_rds_path <- file.path(cds_path, "cds_object.rds")
    if (!file.exists(cds_rds_path)) {
      stop("Could not find cds_object.rds at ", cds_rds_path)
    }
    message("  Reading metadata object only: ", cds_rds_path)
    return(readRDS(cds_rds_path))
  }

  load_monocle_objects(
    cds_path,
    matrix_control = list(matrix_class = "BPCells", matrix_path = temp_path)
  )
}

count_atlas_cells <- function(cds, dataset_name) {
  message("Counting embryo-level cell groups for ", dataset_name)
  md <- as.data.frame(colData(cds))
  sample_col <- first_present(sample_group_candidates, colnames(md), "embryo/sample column")
  time_col <- first_present(timepoint_candidates, colnames(md), "timepoint/hpf column")
  cell_col <- first_present(cell_group_candidates, colnames(md), "cell type column")

  md$hpf_numeric <- parse_hpf(md[[time_col]])
  keep_cells <- rownames(md)[!is.na(md$hpf_numeric) & md$hpf_numeric >= hpf_range[[1]] & md$hpf_numeric <= hpf_range[[2]]]

  if ("perturbation" %in% colnames(md)) {
    keep_cells <- intersect(
      keep_cells,
      rownames(md)[is.na(md$perturbation) | md$perturbation %in% ctrl_labels]
    )
  }

  if (length(keep_cells) == 0) {
    warning("No cells retained for ", dataset_name)
    return(tibble())
  }

  embryo_hpf <- md[keep_cells, , drop = FALSE] %>%
    mutate(embryo = as.character(.data[[sample_col]]), hpf = hpf_numeric) %>%
    filter(!is.na(embryo), !is.na(hpf)) %>%
    group_by(embryo) %>%
    summarise(hpf = median(unique(hpf), na.rm = TRUE), .groups = "drop")

  md[keep_cells, , drop = FALSE] %>%
    mutate(
      embryo = as.character(.data[[sample_col]]),
      cell_group = as.character(.data[[cell_col]]),
      dataset = dataset_name
    ) %>%
    filter(!is.na(embryo), !is.na(cell_group)) %>%
    count(dataset, embryo, cell_group, name = "cell_count") %>%
    left_join(embryo_hpf, by = "embryo") %>%
    filter(!is.na(hpf)) %>%
    mutate(hpf_label = factor(hpf, levels = sort(unique(hpf)))) %>%
    select(dataset, embryo, hpf, hpf_label, cell_group, cell_count)
}

filter_notochord_counts <- function(cell_counts) {
  notochord_counts <- cell_counts %>%
    filter(str_detect(as.character(cell_group), notochord_pattern))

  if (nrow(notochord_counts) == 0) {
    warning("No notochord-related cell groups found in cached cell count table.")
  }

  notochord_counts
}

sanitize_dataset_name <- function(dataset_name) {
  str_replace_all(dataset_name, "[^[:alnum:]_.-]+", "_")
}

dataset_cell_count_cache_path <- function(dataset_name) {
  file.path(count_cache_dir, paste0("embryo_cell_counts_", sanitize_dataset_name(dataset_name), ".rds"))
}

dataset_legacy_cell_count_cache_path <- function(dataset_name) {
  file.path(count_cache_dir, paste0("cell_counts_", sanitize_dataset_name(dataset_name), ".rds"))
}

existing_dataset_cell_count_cache_path <- function(dataset_name) {
  candidates <- c(
    dataset_cell_count_cache_path(dataset_name),
    dataset_legacy_cell_count_cache_path(dataset_name)
  )
  existing <- candidates[file.exists(candidates)]
  if (length(existing) > 0) return(existing[[1]])

  candidates[[1]]
}

dataset_notochord_count_cache_path <- function(dataset_name) {
  file.path(count_cache_dir, paste0("notochord_counts_", sanitize_dataset_name(dataset_name), ".rds"))
}

format_cell_counts <- function(cell_counts) {
  cell_counts %>%
    mutate(
      dataset = factor(dataset, levels = datasets_to_compare),
      cell_group = factor(cell_group, levels = sort(unique(cell_group))),
      hpf_label = factor(hpf, levels = sort(unique(hpf)))
    )
}

write_count_table <- function(count_table, rds_path) {
  csv_path <- str_replace(rds_path, "\\.rds$", ".csv")

  saveRDS(count_table, rds_path)
  write.csv(count_table, csv_path, row.names = FALSE)
  message("Saved: ", rds_path)
  message("Saved: ", csv_path)

  count_table
}

write_dataset_cell_counts <- function(cell_counts, dataset_name) {
  write_count_table(cell_counts, dataset_cell_count_cache_path(dataset_name))
}

write_dataset_notochord_counts <- function(notochord_counts, dataset_name) {
  write_count_table(notochord_counts, dataset_notochord_count_cache_path(dataset_name))
}

read_cached_cell_counts <- function(dataset_names = datasets_to_compare) {
  paths <- setNames(purrr::map_chr(dataset_names, existing_dataset_cell_count_cache_path), dataset_names)
  missing <- names(paths)[!file.exists(paths)]

  if (length(missing) > 0) {
    stop(sprintf(
      "Missing cached embryo-level cell count tables for: %s. Set active_dataset to each missing dataset and run the active atlas count cell.",
      paste(missing, collapse = ", ")
    ))
  }

  purrr::map_dfr(paths, readRDS)
}

read_cached_notochord_counts <- function(dataset_names = datasets_to_compare) {
  cell_counts <- read_cached_cell_counts(dataset_names)
  filter_notochord_counts(cell_counts)
}

format_notochord_counts <- function(notochord_counts) {
  notochord_counts %>%
    format_cell_counts()
  }

save_plot <- function(plot, filename, width = 11, height = 7) {
  pdf_path <- file.path(out_dir, paste0(filename, ".pdf"))
  png_path <- file.path(out_dir, paste0(filename, ".png"))
  ggsave(pdf_path, plot, width = width, height = height, units = "in")
  ggsave(png_path, plot, width = width, height = height, units = "in", dpi = 300)
  message("Saved: ", pdf_path)
  message("Saved: ", png_path)
}

## Load Active Atlas Into Memory

In [ ]:
active_cds_path <- resolve_cds_path(active_dataset)
active_cds_path

if (exists("active_cds")) {
  rm(active_cds)
  gc()
}

active_cds <- load_atlas_cds(active_dataset, metadata_only = load_metadata_only)
active_cds

## Count Active Atlas Embryo Cells

In [ ]:
active_cell_count_cache <- existing_dataset_cell_count_cache_path(active_dataset)

if (!force_recount_cell_counts && file.exists(active_cell_count_cache)) {
  message("Reading cached cell counts: ", active_cell_count_cache)
  active_cell_counts <- readRDS(active_cell_count_cache) %>%
    format_cell_counts()
} else {
  if (!exists("active_cds")) {
    stop("active_cds is not loaded and no cell count cache exists. Run the active atlas load cell first, or set active_dataset to a cached dataset.")
  }

  active_cell_counts <- count_atlas_cells(active_cds, active_dataset) %>%
    format_cell_counts()

  if (nrow(active_cell_counts) == 0) {
    stop("No cell count data found for active_dataset. Check dataset path and cell type labels.")
  }

  write_dataset_cell_counts(active_cell_counts, active_dataset)
}

if (nrow(active_cell_counts) == 0) {
  stop("No cell count data found for active_dataset. Check dataset path and cell type labels.")
}

active_notochord_counts <- active_cell_counts %>%
  filter_notochord_counts() %>%
  format_notochord_counts()

if (nrow(active_notochord_counts) == 0) {
  stop("No notochord count data found for active_dataset. Check cell type labels.")
}

write_dataset_notochord_counts(active_notochord_counts, active_dataset)

# active_notochord_counts %>%
#   count(dataset, cell_group, hpf, name = "n_embryos") %>%
#   arrange(cell_group, dataset, hpf)

## Free Active Atlas Memory

In [ ]:
if (exists("active_cds")) {
  rm(active_cds)
  gc()
}

## Combine Cached Counts

In [ ]:
cell_counts <- read_cached_cell_counts(datasets_to_compare) %>%
  format_cell_counts()

write.csv(
  cell_counts,
  file.path(out_dir, "embryo_cell_counts_long.csv"),
  row.names = FALSE
)

write.csv(
  dataset_specs,
  file.path(out_dir, "dataset_specs.csv"),
  row.names = FALSE
)

embryo_total_cells <- cell_counts %>%
  mutate(dataset = as.character(dataset)) %>%
  group_by(dataset, embryo, hpf, hpf_label) %>%
  summarise(total_cells = sum(cell_count, na.rm = TRUE), .groups = "drop")

cell_group_key_template <- tribble(
  ~harmonization_family, ~cell_group, ~standardized_cell_group,
  "v2.3.0", "hypochord", "hypochord",
  "v2.3.0", "notochord (suspected doublets)", NA_character_,
  "v2.3.0", "notochordal cell", "notochordal cell",
  "v2.3.0", "notochordal cell (noto+)", "notochordal cell",
  "v2.3.0", "notochordal sheath cell (G2_M)", "notochordal sheath cell",
  "v2.3.0", "notochordal sheath cell (late)", "notochordal sheath cell",
  "v2.3.0", "notochordal sheath cell (late, G2_M)", "notochordal sheath cell",
  "v2.3.0", "notochordal sheath cell (late, entpd5a+)", "notochordal sheath cell",
  "v2.3.0", "notochordal vacuole cell (early)", "notochordal vacuole cell",
  "v2.3.0", "notochordal vacuole cell (late)", "notochordal vacuole cell",
  "v3.1.0", "notochordal cell, notochord (11-18 hpf)", "notochordal cell",
  "v3.1.0", "notochordal cell, notochord (18 hpf)", "notochordal cell",
  "v3.1.0", "notochordal cell, notochord (posterior, 11-18 hpf)", "notochordal cell",
  "v3.1.0", "notochordal sheath cell, notochord", "notochordal sheath cell",
  "v3.1.0", "notochordal vacuole cell, notochord (18 hpf)", "notochordal vacuole cell",
  "v3.1.0", "notochordal vacuole cell, notochord (24-96 hpf)", "notochordal vacuole cell",
  "v3.1.0", "unknown, hypochord", "hypochord"
)

cell_group_key <- dataset_specs %>%
  select(dataset, harmonization_family) %>%
  inner_join(cell_group_key_template, by = "harmonization_family") %>%
  select(dataset, cell_group, standardized_cell_group)

raw_notochord_counts <- cell_counts %>%
  filter_notochord_counts() %>%
  mutate(dataset = as.character(dataset), cell_group = as.character(cell_group))

dropped_cell_groups <- raw_notochord_counts %>%
  distinct(dataset, cell_group) %>%
  left_join(cell_group_key, by = c("dataset", "cell_group")) %>%
  filter(is.na(standardized_cell_group) | standardized_cell_group == "") %>%
  arrange(dataset, cell_group)

write.csv(
  dropped_cell_groups,
  file.path(out_dir, "dropped_notochord_cell_groups.csv"),
  row.names = FALSE
)

observed_notochord_counts <- raw_notochord_counts %>%
  left_join(cell_group_key, by = c("dataset", "cell_group")) %>%
  filter(!is.na(standardized_cell_group), standardized_cell_group != "") %>%
  group_by(dataset, embryo, hpf, hpf_label, cell_group = standardized_cell_group) %>%
  summarise(cell_count = sum(cell_count, na.rm = TRUE), .groups = "drop")

embryo_stage_grid <- cell_counts %>%
  distinct(dataset = as.character(dataset), embryo, hpf, hpf_label)

standardized_cell_groups <- cell_group_key %>%
  filter(!is.na(standardized_cell_group), standardized_cell_group != "") %>%
  distinct(cell_group = standardized_cell_group) %>%
  arrange(cell_group)

notochord_counts <- embryo_stage_grid %>%
  tidyr::crossing(standardized_cell_groups) %>%
  left_join(
    observed_notochord_counts,
    by = c("dataset", "embryo", "hpf", "hpf_label", "cell_group")
  ) %>%
  left_join(
    embryo_total_cells,
    by = c("dataset", "embryo", "hpf", "hpf_label")
  ) %>%
  mutate(
    cell_count = tidyr::replace_na(cell_count, 0),
    cell_count_per_1000 = if_else(total_cells > 0, 1000 * cell_count / total_cells, NA_real_)
  ) %>%
  format_notochord_counts()

zero_completion_summary <- notochord_counts %>%
  group_by(dataset, hpf, cell_group) %>%
  summarise(
    n_embryos = n(),
    n_zero = sum(cell_count == 0),
    frac_zero = n_zero / n_embryos,
    .groups = "drop"
  )

plot_notochord_counts <- notochord_counts %>%
  filter(hpf %in% plot_hpf_values) %>%
  format_notochord_counts()

write.csv(
  raw_notochord_counts,
  file.path(out_dir, "raw_notochord_cell_counts_long.csv"),
  row.names = FALSE
)

write.csv(
  notochord_counts,
  file.path(out_dir, "notochord_cell_counts_completed_long.csv"),
  row.names = FALSE
)

write.csv(
  zero_completion_summary,
  file.path(out_dir, "zero_completion_summary.csv"),
  row.names = FALSE
)

write.csv(
  plot_notochord_counts,
  file.path(out_dir, "notochord_cell_counts_long.csv"),
  row.names = FALSE
)

write.csv(
  plot_notochord_counts %>%
    select(dataset, embryo, hpf, hpf_label, cell_group, total_cells, cell_count_per_1000),
  file.path(out_dir, "notochord_cell_counts_per_1000_long.csv"),
  row.names = FALSE
)

zero_completion_summary %>%
  filter(hpf %in% plot_hpf_values) %>%
  arrange(dataset, hpf, cell_group)

In [ ]:
print(cell_group_key, n = Inf)

write.csv(
  cell_group_key,
  file.path(out_dir, "cell_group_key.csv"),
  row.names = FALSE
)

dropped_cell_groups

## Per-Cell-Type Counts

In [ ]:
boxplot_stat_table <- function(data, value_col, group_cols) {
  data %>%
    group_by(across(all_of(group_cols))) %>%
    summarise(
      box_stats = list(boxplot.stats(.data[[value_col]])$stats),
      .groups = "drop"
    ) %>%
    mutate(
      ymin = purrr::map_dbl(box_stats, 1),
      lower = purrr::map_dbl(box_stats, 2),
      middle = purrr::map_dbl(box_stats, 3),
      upper = purrr::map_dbl(box_stats, 4),
      ymax = purrr::map_dbl(box_stats, 5)
    ) %>%
    select(-box_stats)
}

boxplot_theme <- theme(
  plot.title.position = "plot",
  plot.title = element_text(size = 30, margin = margin(b = 8)),
  plot.subtitle = element_text(size = 20, margin = margin(b = 12)),
  plot.margin = margin(t = 24, r = 26, b = 22, l = 26),
  legend.position = "bottom",
  legend.box = "vertical",
  legend.key.width = grid::unit(1.2, "lines"),
  legend.text = element_text(size = 16, lineheight = 0.9),
  legend.title = element_text(size = 18),
  axis.title = element_text(size = 18),
  axis.text = element_text(size = 13),
  axis.text.x = element_text(size = 12, angle = 45, hjust = 1),
  axis.title.y = element_text(margin = margin(r = 10)),
  axis.title.x = element_text(margin = margin(t = 10)),
  panel.grid.minor = element_blank(),
  strip.text = element_text(size = 17, lineheight = 0.95)
)

wrap_plot_labels <- function(width = 16) {
  function(x) stringr::str_wrap(x, width = width)
}

cell_boxplot_stats <- boxplot_stat_table(
  plot_notochord_counts,
  value_col = "cell_count",
  group_cols = c("dataset", "cell_group", "hpf", "hpf_label")
)

p_cell_counts <- ggplot(cell_boxplot_stats, aes(x = hpf_label, fill = dataset)) +
  geom_boxplot(
    aes(ymin = ymin, lower = lower, middle = middle, upper = upper, ymax = ymax),
    stat = "identity",
    position = position_dodge2(width = 0.8, preserve = "single"),
    width = 0.65
  ) +
  facet_wrap(~ cell_group, scales = "free_y", labeller = labeller(cell_group = label_wrap_gen(width = 18))) +
  scale_fill_brewer(palette = "Set2", drop = FALSE, labels = wrap_plot_labels(16)) +
  guides(fill = guide_legend(nrow = 2, byrow = TRUE)) +
  labs(
    title = str_wrap("Notochord-related cell counts per embryo", width = 85),
    subtitle = str_wrap(sprintf("%s hpf; harmonized labels only", plot_hpf_label), width = 95),
    x = "hpf",
    y = "Cells\nper embryo",
    fill = "Atlas"
  ) +
  boxplot_theme
options(repr.plot.width = 24, repr.plot.height = 16, repr.plot.res = 140)
print(p_cell_counts)
save_plot(p_cell_counts, "notochord_cell_counts_by_atlas", width = 26, height = 16)

cell_per_1000_boxplot_stats <- boxplot_stat_table(
  plot_notochord_counts,
  value_col = "cell_count_per_1000",
  group_cols = c("dataset", "cell_group", "hpf", "hpf_label")
)

p_cell_counts_per_1000 <- ggplot(cell_per_1000_boxplot_stats, aes(x = hpf_label, fill = dataset)) +
  geom_boxplot(
    aes(ymin = ymin, lower = lower, middle = middle, upper = upper, ymax = ymax),
    stat = "identity",
    position = position_dodge2(width = 0.8, preserve = "single"),
    width = 0.65
  ) +
  facet_wrap(~ cell_group, scales = "free_y", labeller = labeller(cell_group = label_wrap_gen(width = 18))) +
  scale_fill_brewer(palette = "Set2", drop = FALSE, labels = wrap_plot_labels(16)) +
  guides(fill = guide_legend(nrow = 2, byrow = TRUE)) +
  labs(
    title = str_wrap("Notochord-related cells per 1,000 cells", width = 85),
    subtitle = str_wrap(sprintf("%s hpf; harmonized labels only", plot_hpf_label), width = 95),
    x = "hpf",
    y = "Cells per 1,000\nrecovered cells",
    fill = "Atlas"
  ) +
  boxplot_theme

print(p_cell_counts_per_1000)
save_plot(p_cell_counts_per_1000, "notochord_cell_counts_per_1000_by_atlas", width = 26, height = 16)

## Total Notochord-Related Counts

In [ ]:
# Collection timepoints for the planned experiment. Total-cell depth is the
# quantity that drives NB power, so this panel is restricted to the stages we
# actually collect rather than the broader plot_hpf_values sweep.
depth_hpf_values <- c(24, 30, 36)
depth_hpf_label <- paste(depth_hpf_values, collapse = ", ")

plot_embryo_total_cells <- embryo_total_cells %>%
  mutate(
    dataset = factor(dataset, levels = datasets_to_compare),
    hpf_label = factor(hpf, levels = sort(unique(hpf)))
  )

write.csv(
  plot_embryo_total_cells,
  file.path(out_dir, "total_cells_by_embryo.csv"),
  row.names = FALSE
)

depth_plot_data <- plot_embryo_total_cells %>%
  filter(hpf %in% depth_hpf_values) %>%
  mutate(hpf_label = factor(hpf, levels = sort(depth_hpf_values)))

# Drop dataset/stage combinations with too few embryos to box meaningfully.
depth_group_n <- depth_plot_data %>%
  count(dataset, hpf, hpf_label, name = "n_embryos")

depth_plot_data <- depth_plot_data %>%
  semi_join(depth_group_n %>% filter(n_embryos >= 3), by = c("dataset", "hpf", "hpf_label"))

total_cells_boxplot_stats <- boxplot_stat_table(
  depth_plot_data,
  value_col = "total_cells",
  group_cols = c("dataset", "hpf", "hpf_label")
) %>%
  left_join(depth_group_n, by = c("dataset", "hpf", "hpf_label"))

p_total_atlas_cells <- ggplot(total_cells_boxplot_stats, aes(x = hpf_label, fill = dataset)) +
  geom_boxplot(
    aes(ymin = ymin, lower = lower, middle = middle, upper = upper, ymax = ymax),
    stat = "identity",
    position = position_dodge2(width = 0.8, preserve = "single"),
    width = 0.65
  ) +
  geom_text(
    aes(y = ymax, label = paste0("n=", n_embryos)),
    position = position_dodge2(width = 0.8, preserve = "single"),
    vjust = -0.6,
    size = 3.6,
    color = "grey25"
  ) +
  scale_fill_brewer(palette = "Set2", drop = FALSE, labels = wrap_plot_labels(16)) +
  scale_y_continuous(expand = expansion(mult = c(0.02, 0.12)), labels = scales::comma) +
  guides(fill = guide_legend(nrow = 2, byrow = TRUE)) +
  labs(
    title = "Total recovered cells per embryo",
    subtitle = sprintf("%s hpf; one box per atlas/experiment and stage", depth_hpf_label),
    x = "hpf",
    y = "Total recovered\ncells per embryo",
    fill = "Atlas / experiment"
  ) +
  boxplot_theme +
  theme(
    plot.title = element_text(size = 20, margin = margin(b = 6)),
    plot.subtitle = element_text(size = 14, margin = margin(b = 10)),
    axis.title = element_text(size = 14),
    axis.text = element_text(size = 12),
    axis.text.x = element_text(size = 12, angle = 0, hjust = 0.5),
    legend.text = element_text(size = 11, lineheight = 0.9),
    legend.title = element_text(size = 12)
  )

options(repr.plot.width = 13, repr.plot.height = 8, repr.plot.res = 140)
print(p_total_atlas_cells)
save_plot(p_total_atlas_cells, "total_cells_per_embryo_by_atlas", width = 13, height = 8)

# Median depth per group, for reference alongside the plot.
depth_plot_data %>%
  group_by(dataset, hpf) %>%
  summarise(
    n_embryos = n(),
    median_total_cells = median(total_cells),
    mean_total_cells = round(mean(total_cells)),
    .groups = "drop"
  ) %>%
  arrange(dataset, hpf)

notochord_totals <- plot_notochord_counts %>%
  group_by(dataset, embryo, hpf, hpf_label) %>%
  summarise(
    total_notochord_cells = sum(cell_count, na.rm = TRUE),
    total_cells = first(total_cells),
    total_notochord_cells_per_1000 = if_else(total_cells > 0, 1000 * total_notochord_cells / total_cells, NA_real_),
    .groups = "drop"
  )

write.csv(
  notochord_totals,
  file.path(out_dir, "notochord_total_counts_by_embryo.csv"),
  row.names = FALSE
)

write.csv(
  notochord_totals %>%
    select(dataset, embryo, hpf, hpf_label, total_cells, total_notochord_cells_per_1000),
  file.path(out_dir, "notochord_total_counts_per_1000_by_embryo.csv"),
  row.names = FALSE
)

total_boxplot_stats <- boxplot_stat_table(
  notochord_totals,
  value_col = "total_notochord_cells",
  group_cols = c("dataset", "hpf", "hpf_label")
)

p_total_counts <- ggplot(total_boxplot_stats, aes(x = hpf_label, fill = dataset)) +
  geom_boxplot(
    aes(ymin = ymin, lower = lower, middle = middle, upper = upper, ymax = ymax),
    stat = "identity",
    position = position_dodge2(width = 0.8, preserve = "single"),
    width = 0.65
  ) +
  scale_fill_brewer(palette = "Set2", drop = FALSE, labels = wrap_plot_labels(16)) +
  guides(fill = guide_legend(nrow = 2, byrow = TRUE)) +
  labs(
    title = str_wrap("Total notochord-related cells per embryo", width = 85),
    subtitle = str_wrap(sprintf("%s hpf; harmonized labels only", plot_hpf_label), width = 95),
    x = "hpf",
    y = "Notochord-related\ncells per embryo",
    fill = "Atlas"
  ) +
  boxplot_theme

print(p_total_counts)
save_plot(p_total_counts, "notochord_total_counts_by_atlas", width = 16, height = 9)

total_per_1000_boxplot_stats <- boxplot_stat_table(
  notochord_totals,
  value_col = "total_notochord_cells_per_1000",
  group_cols = c("dataset", "hpf", "hpf_label")
)

p_total_counts_per_1000 <- ggplot(total_per_1000_boxplot_stats, aes(x = hpf_label, fill = dataset)) +
  geom_boxplot(
    aes(ymin = ymin, lower = lower, middle = middle, upper = upper, ymax = ymax),
    stat = "identity",
    position = position_dodge2(width = 0.8, preserve = "single"),
    width = 0.65
  ) +
  scale_fill_brewer(palette = "Set2", drop = FALSE, labels = wrap_plot_labels(16)) +
  guides(fill = guide_legend(nrow = 2, byrow = TRUE)) +
  labs(
    title = str_wrap("Total notochord-related cells per 1,000 cells", width = 85),
    subtitle = str_wrap(sprintf("%s hpf; harmonized labels only", plot_hpf_label), width = 95),
    x = "hpf",
    y = "Notochord-related cells\nper 1,000 recovered cells",
    fill = "Atlas"
  ) +
  boxplot_theme

print(p_total_counts_per_1000)
save_plot(p_total_counts_per_1000, "notochord_total_counts_per_1000_by_atlas", width = 16, height = 9)

## Summary

In [ ]:
summary_table <- plot_notochord_counts %>%
  group_by(dataset, cell_group, hpf) %>%
  summarise(
    n_embryos = n(),
    mean_count = mean(cell_count, na.rm = TRUE),
    median_count = median(cell_count, na.rm = TRUE),
    mean_count_per_1000 = mean(cell_count_per_1000, na.rm = TRUE),
    median_count_per_1000 = median(cell_count_per_1000, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(cell_group, dataset, hpf)

write.csv(summary_table, file.path(out_dir, "notochord_count_summary.csv"), row.names = FALSE)
summary_table